# Felix Baseline CatBoost Notebook

This is a baseline CatBoost workflow that demonstrates the core portfolio_toolkit pattern.

## Workflow Summary

1. Load shared dataset (shared_set_2)
2. Build features: shared toolkit features + 1 custom feature
3. Create targets
4. Train CatBoost model for expected returns
5. Generate predictions
6. Convert to portfolio weights
7. Run shared backtest
8. Log results to MLflow

In [1]:
# Setup and imports
from pathlib import Path
import numpy as np
import pandas as pd

from portfolio_toolkit import (
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_risk_adjusted,
    backtest_weights,
    write_backtest_artifacts,
)

print('Imports loaded successfully.')

/Users/beizhang/Portfolio-Optimization-Lib/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports loaded successfully.


In [3]:
# Configuration
# Find repo root by looking for configs/datasets.toml
from pathlib import Path
import sys

current_dir = Path.cwd()
repo_root = None

# Walk up the directory tree to find repo_root
for parent in [current_dir] + list(current_dir.parents):
    if (parent / 'configs' / 'datasets.toml').exists():
        repo_root = parent
        break

if repo_root is None:
    raise FileNotFoundError("Could not find repo_root. Make sure you run this from the Portfolio-Optimization-Lib repo.")

dataset_name = 'shared_set_2'
model_name = 'felix_baseline_catboost'
horizon = 5
output_dir = repo_root / 'runs' / 'felix_baseline_catboost'
output_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(dataset_name, repo_root=repo_root)
splits = split_dates(dataset_name, repo_root=repo_root)

print('Repo root:', repo_root)
print('Dataset preset:', dataset_name)
print('Dataset name:', spec.name)
print('Tickers:', len(spec.tickers))
print('Splits:', splits)

Repo root: /Users/beizhang/Portfolio-Optimization-Lib
Dataset preset: shared_set_2
Dataset name: growth_tech_innovation
Tickers: 26
Splits: {'train': (Timestamp('2014-01-02 00:00:00'), Timestamp('2019-12-31 00:00:00')), 'val': (Timestamp('2020-01-02 00:00:00'), Timestamp('2021-12-31 00:00:00')), 'test': (Timestamp('2022-01-03 00:00:00'), Timestamp('2025-12-31 00:00:00'))}


## 1. Load Shared Price Data

In [ ]:
prices = load_prices(dataset_name, repo_root=repo_root)

print('Price frame shape:', prices.shape)
print('Date range:', prices['date'].min(), '->', prices['date'].max())
print('Unique tickers:', prices['ticker'].nunique())
display(prices.head())

## 2. Build Features: Shared + Custom

In [ ]:
# Build shared baseline features
base_feature_names = [
    'return_5d',
    'return_20d',
    'vol_20d',
    'momentum_20d',
    'momentum_60d',
    'rsi_14',
    'price_to_sma_20d',
    'price_to_sma_50d',
    'volume_zscore_20d',
    'excess_return_20d_vs_spy',
]

features = build_features(prices, feature_names=base_feature_names)
print('Base feature frame shape:', features.shape)
display(features.head())

In [ ]:
# Add one custom notebook-local feature
# momentum_vol_ratio: captures risk-adjusted trend strength
eps = 1e-6
features['momentum_vol_ratio'] = features['momentum_20d'] / (features['vol_20d'].abs() + eps)

feature_names = base_feature_names + ['momentum_vol_ratio']

print('Feature frame shape with custom feature:', features.shape)
print('Feature count:', len(feature_names))
display(features[['date', 'ticker', 'momentum_vol_ratio']].head())

## 3. Build Targets

In [ ]:
# Build forward return target only (baseline model)
targets = make_forward_return_target(prices, horizon=horizon)

# Merge features and targets
modeling_frame = features.merge(targets, on=['date', 'ticker'], how='left')

# Drop rows with missing values
modeling_frame = modeling_frame.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

target_col = f'forward_return_{horizon}d'

print('Modeling frame shape after dropping nulls:', modeling_frame.shape)
display(modeling_frame.head())

## 4. Train/Val/Test Split and Standardization

In [ ]:
# Use shared split functions
train = slice_split(modeling_frame, dataset_name, 'train', repo_root=repo_root)
val = slice_split(modeling_frame, dataset_name, 'val', repo_root=repo_root)
test = slice_split(modeling_frame, dataset_name, 'test', repo_root=repo_root)

print(f'Train: {len(train)} rows')
print(f'Val: {len(val)} rows')
print(f'Test: {len(test)} rows')

# Standardize using only train statistics
train_means = train[feature_names].mean()
train_stds = train[feature_names].std(ddof=0).replace(0.0, 1.0)

def standardize(df):
    return ((df[feature_names] - train_means) / train_stds).to_numpy(dtype=float)

X_train = standardize(train)
X_val = standardize(val)
X_test = standardize(test)

y_train = train[target_col].to_numpy(dtype=float)
y_val = val[target_col].to_numpy(dtype=float)

print(f'X_train shape: {X_train.shape}')
print(f'Feature count: {len(feature_names)}')

## 5. Train CatBoost Model

In [ ]:
from catboost import CatBoostRegressor

# Train baseline CatBoost model
model = CatBoostRegressor(
    iterations=100,
    learning_rate=0.05,
    random_seed=42,
    verbose=False,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=20
)

val_mse = float(((model.predict(X_val) - y_val) ** 2).mean())
val_rmse = float(val_mse ** 0.5)

print(f'Validation MSE: {val_mse:.8f}')
print(f'Validation RMSE: {val_rmse:.8f}')

## 6. Generate Predictions

In [ ]:
# Generate test predictions
test_predictions = model.predict(X_test)

# Create standardized prediction frame
predictions = test.loc[:, ['date', 'ticker']].copy()
predictions['horizon'] = horizon
predictions['expected_return'] = test_predictions
predictions['uncertainty'] = val_rmse  # Use validation RMSE as uncertainty estimate

# Validate prediction frame
predictions = validate_prediction_frame(
    predictions,
    dataset_name=dataset_name,
    horizon=horizon,
    repo_root=repo_root,
)

print('Prediction frame shape:', predictions.shape)
display(predictions.head())

## 7. Convert to Portfolio Weights

In [ ]:
# Use simple risk-adjusted weighting
# Clip expected_return to avoid extreme weights
predictions_clipped = predictions.copy()
predictions_clipped['expected_return'] = predictions_clipped['expected_return'].clip(-0.5, 0.5)

# Create a simple volatility estimate for weighting
predictions_clipped['expected_volatility'] = 0.15  # Fixed volatility baseline

# Generate portfolio weights
portfolio = weights_from_predictions_risk_adjusted(
    predictions_clipped,
    dataset_name=dataset_name,
    strategy_name=model_name,
)

# Validate weights
validated_weights = validate_weights_frame(
    portfolio.weights,
    dataset_name=dataset_name,
    repo_root=repo_root,
)

print('Strategy:', portfolio.strategy_name)
print('Weights frame shape:', validated_weights.shape)
display(validated_weights.head())

## 8. Run Shared Backtest

In [ ]:
# Run the shared backtest
result = backtest_weights(dataset_name, portfolio, repo_root=repo_root)

# Build metrics
metrics = build_metrics(result)

# Write artifacts
artifact_paths = write_backtest_artifacts(result, output_dir)

print('Backtest completed successfully')
print('Metrics:')
for key, value in sorted(metrics.items()):
    print(f'  {key}: {value:.6f}')

## 9. Log to MLflow

In [ ]:
import mlflow

mlflow_layout = init_mlflow(repo_root)
print('MLflow tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name='Felix_Baseline_CatBoost',
    dataset_name=dataset_name,
    tags={
        'workflow': 'felix_baseline',
        'model_family': 'catboost',
        'prediction_horizon': str(horizon),
    },
    repo_root=repo_root,
):
    mlflow.log_params({
        'model_name': model_name,
        'dataset_name': dataset_name,
        'horizon': horizon,
        'feature_count': len(feature_names),
        'catboost_iterations': 100,
        'catboost_lr': 0.05,
        'catboost_seed': 42,
        'custom_features': 'momentum_vol_ratio',
    })
    
    mlflow.log_metrics({
        'val_mse': val_mse,
        'val_rmse': val_rmse,
    })
    
    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)

print('MLflow run Felix_Baseline_CatBoost logged successfully')

## 10. Results Summary

In [ ]:
print('=== BASELINE CATBOOST RESULTS ===')
print()
print('Model Performance:')
for key, value in sorted(metrics.items()):
    print(f'  {key}: {value:.6f}')

print()
print('Backtest Output:')
display(result.nav.tail().to_frame('nav'))
display(result.returns.tail().to_frame('returns'))
display(result.turnover.tail().to_frame('turnover'))

print()
print('Artifacts:')
for key, path in artifact_paths.items():
    print(f'  {key}: {path}')

In [ ]:
# Final validation
assert 'sharpe' in result.metrics, 'Sharpe ratio not computed'
assert 'max_drawdown' in result.metrics, 'Max drawdown not computed'
assert (validated_weights.sum(axis=1).round(6) == 1.0).all(), 'Weights do not sum to 1.0'
assert len(predictions) > 0, 'No predictions generated'
print('✓ All validations passed')
print('✓ Baseline CatBoost workflow completed successfully')